# 🇺🇸 Obama Speeches — Development of America
### Comprehensive Text & Web Analytics — Full Syllabus Coverage

| Topic | Coverage |
|---|---|
| Text Preprocessing | Tokenization, Stemming, Lemmatization, Stop Words |
| Text Mining | BoW, TF-IDF, N-grams, Word2Vec, GloVe |
| Classification | Naive Bayes, SVM, Logistic Regression |
| Sentiment Analysis | VADER Rule-Based + ML, Aspect-Based |
| Topic Modelling | LDA, LSA, K-Means, Hierarchical Clustering |
| NER | Rule-Based + spaCy ML-Based |
| Summarization | Extractive (TextRank) + Abstractive |
| Web Scraping | BeautifulSoup, robots.txt, Ethics |
| Web Analytics | KPIs, Traffic Metrics |
| Business Intelligence | Competitive Intelligence, Risk, Personalization |

> All plots are **automatically saved** to an `obama_plots/` folder on your machine.
---

In [2]:
# ══════════════════════════════════════════════════════
# CELL 0 ▸ SUPPRESS ALL WARNINGS  ← Run this first!
# ══════════════════════════════════════════════════════
import warnings, os, sys, logging
warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
for _lg in ['gensim','sklearn','matplotlib','urllib3']:
    logging.getLogger(_lg).setLevel(logging.ERROR)
print('✅  Warnings suppressed.')

✅  Warnings suppressed.


In [3]:
# ══════════════════════════════════════════════════════
# CELL 1 ▸ INSTALL ALL LIBRARIES
# ══════════════════════════════════════════════════════
import subprocess, sys
PACKAGES = [
    'requests','beautifulsoup4','pandas','numpy','scikit-learn',
    'matplotlib','seaborn','networkx','spacy','vaderSentiment',
    'gensim','nltk','wordcloud','scipy'
]
print('Installing / verifying libraries...\n')
for pkg in PACKAGES:
    r = subprocess.run([sys.executable,'-m','pip','install',pkg,'-q'],
                       capture_output=True, text=True)
    print(f"  {'✅' if r.returncode==0 else '❌'}  {pkg}")
subprocess.run([sys.executable,'-m','spacy','download','en_core_web_sm','-q'],
               capture_output=True)
print('  ✅  spaCy: en_core_web_sm')
import nltk
for res in ['punkt','stopwords','averaged_perceptron_tagger',
            'maxent_ne_chunker','words','wordnet','vader_lexicon','punkt_tab']:
    nltk.download(res, quiet=True)
print('  ✅  NLTK corpora\n\n✅  All ready!')

Installing / verifying libraries...

  ✅  requests
  ✅  beautifulsoup4
  ✅  pandas
  ✅  numpy
  ✅  scikit-learn
  ✅  matplotlib
  ✅  seaborn
  ✅  networkx
  ✅  spacy
  ✅  vaderSentiment
  ✅  gensim
  ✅  nltk
  ✅  wordcloud
  ✅  scipy
  ✅  spaCy: en_core_web_sm
  ✅  NLTK corpora

✅  All ready!


In [4]:
# ══════════════════════════════════════════════════════
# CELL 2 ▸ IMPORTS + GLOBAL SETTINGS
# ══════════════════════════════════════════════════════
import re, time, random, urllib.robotparser
from io import StringIO
from collections import Counter, defaultdict
from urllib.parse import urljoin

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup

import spacy
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.corpus import stopwords as nltk_sw
from nltk.tag import pos_tag
from nltk.chunk import ne_chunk

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score
)
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import TruncatedSVD
from sklearn.decomposition import LatentDirichletAllocation as SklearnLDA
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import TSNE
from sklearn.preprocessing import LabelEncoder

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

import gensim
import gensim.corpora as corpora
from gensim.models import LdaModel, Word2Vec

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as cm
import seaborn as sns
from wordcloud import WordCloud
from scipy.cluster.hierarchy import dendrogram, linkage
import networkx as nx

# ── spaCy ──
try:
    nlp = spacy.load('en_core_web_sm')
    print('✅  spaCy loaded')
except:
    print('❌  spaCy model missing')

# ── Plot style ──
plt.rcParams.update({
    'figure.facecolor':'#0f0f1a','axes.facecolor':'#1a1a2e',
    'axes.edgecolor':'#444466','axes.labelcolor':'#ccccdd',
    'axes.titlecolor':'#ffffff','xtick.color':'#aaaacc',
    'ytick.color':'#aaaacc','text.color':'#ccccdd',
    'grid.color':'#2a2a4a','grid.alpha':0.4
})
PALETTE = ['#e040fb','#40c4ff','#69ff47','#ffd740','#ff6e40',
           '#80deea','#ff4081','#b2ff59','#ea80fc','#ffab40']

# ── Output folder for all saved plots ──
PLOT_DIR = 'obama_plots'
os.makedirs(PLOT_DIR, exist_ok=True)

def save_plot(filename):
    """
    Saves the current matplotlib figure to the obama_plots/ folder
    AND displays it inline in the notebook.
    """
    full_path = os.path.join(PLOT_DIR, filename)
    plt.savefig(full_path, dpi=150, bbox_inches='tight',
                facecolor='#0f0f1a')
    plt.show()
    print(f'  💾  Saved → {os.path.abspath(full_path)}')

print(f'✅  Plots will be saved to: {os.path.abspath(PLOT_DIR)}/')
print('✅  All imports complete.')

✅  spaCy loaded
✅  Plots will be saved to: C:\Users\Thanooj Kumar G\obama_plots/
✅  All imports complete.


In [5]:
# ══════════════════════════════════════════════════════
# CELL 3 ▸ UPLOAD CSV FROM YOUR COMPUTER
# ══════════════════════════════════════════════════════
def load_csv():
    """Prompts user to upload CSV — works in Colab and local Jupyter."""
    try:
        from google.colab import files
        print('📂  Click the Upload button below to upload your CSV...')
        uploaded = files.upload()
        fname = list(uploaded.keys())[0]
        df = pd.read_csv(StringIO(uploaded[fname].decode('utf-8')))
        print(f'\n✅  File uploaded: {fname}')
        return df
    except ImportError:
        path = input('📂  Enter full path to your CSV: ').strip()
        df = pd.read_csv(path)
        print(f'\n✅  File loaded: {path}')
        return df

df_raw = load_csv()

# ── Clean & prepare ──
df_raw = df_raw.dropna(subset=['content']).reset_index(drop=True)
df_raw['document_date'] = pd.to_datetime(df_raw['document_date'], errors='coerce')
df_raw['year'] = df_raw['document_date'].dt.year
df_raw['content'] = df_raw['content'].astype(str)
df_raw['title']   = df_raw['title'].astype(str)

print(f'\n  Rows × Columns   : {df_raw.shape}')
print(f'  Date range       : {int(df_raw["year"].min())} – {int(df_raw["year"].max())}')
print(f'  Columns          : {list(df_raw.columns)}')
display(df_raw[['title','year','document_type_name','content']].head(3))


✅  File loaded: D:\MBA Thanooj\SEM 4\Text Analytics\obama-white-house.csv\obama-white-house.csv

  Rows × Columns   : (20317, 15)
  Date range       : 2009 – 2017
  Columns          : ['number', 'name', 'image', 'previous_office', 'presidency_url', 'party_affiliation', 'start_date', 'end_date', 'document_type_name', 'document_types_slug', 'title', 'url', 'content', 'document_date', 'year']


,title,year,document_type_name,content
0,FACT SHEET: United States – Argentina Relatio...,2016,Statements and Releases,"President Obama, accompanied by First Lady Mic..."
1,FACT SHEET: U.S.-China Economic Relations,2015,Statements and Releases,The United States and China recognize their sh...
2,Presidential Memorandum -- Establishing Polici...,2012,Presidential Memoranda,MEMORANDUM FOR THE HEADS OF EXECUTIVE DEPARTME...


In [6]:
# ══════════════════════════════════════════════════════
# CELL 4 ▸ FILTER — DEVELOPMENT OF AMERICA SPEECHES
# ══════════════════════════════════════════════════════
DEV_KW = [
    'economy','jobs','infrastructure','education','healthcare',
    'middle class','growth','investment','innovation','opportunity',
    'energy','manufacturing','small business','workforce','climate',
    'development','employment','trade','technology','poverty'
]
mask = (
    df_raw['content'].str.lower().apply(lambda t: any(k in t for k in DEV_KW))
    | df_raw['title'].str.lower().apply(lambda t: any(k in t for k in DEV_KW))
)
df = df_raw[mask].reset_index(drop=True)

print(f'  Total speeches              : {len(df_raw):,}')
print(f'  Development speeches kept   : {len(df):,}')
print(f'  Filtered out                : {len(df_raw)-len(df):,}')
print(f'\n  Keywords used:')
print(f'  {DEV_KW}')
print(f'\n  First 10 selected speeches:')
for i, row in df.head(10).iterrows():
    yr = int(row['year']) if not pd.isna(row['year']) else 'N/A'
    print(f"  [{i+1:>2}] ({yr})  {row['title'][:72]}")

  Total speeches              : 20,317
  Development speeches kept   : 13,405
  Filtered out                : 6,912

  Keywords used:
  ['economy', 'jobs', 'infrastructure', 'education', 'healthcare', 'middle class', 'growth', 'investment', 'innovation', 'opportunity', 'energy', 'manufacturing', 'small business', 'workforce', 'climate', 'development', 'employment', 'trade', 'technology', 'poverty']

  First 10 selected speeches:
  [ 1] (2016)  FACT SHEET:  United States – Argentina Relationship
  [ 2] (2015)  FACT SHEET: U.S.-China Economic Relations
  [ 3] (2012)  Presidential Memorandum -- Establishing Policies for Addressing Domestic
  [ 4] (2014)  Weekly Address: Reducing Carbon Pollution in Our Power Plants
  [ 5] (2011)  Presidential Memorandum -- Accelerating Technology Transfer and Commerci
  [ 6] (2014)  Weekly Address: Paying Tribute to our Fallen Heroes this Memorial Day
  [ 7] (2014)  Weekly Address: Working When Congress Won’t Act
  [ 8] (2014)  Weekly Address: The First L

---
## 📊 Structured vs Unstructured Data
> **Syllabus:** Definition · Differences · Applications in Business

In [7]:
# ══════════════════════════════════════════════════════
# CELL 5 ▸ STRUCTURED vs UNSTRUCTURED DATA
# ══════════════════════════════════════════════════════
print('─'*60)
print('STRUCTURED COLUMNS — numeric / date / categorical')
print('─'*60)
for c in ['number','start_date','end_date','document_date','year']:
    if c in df.columns:
        print(f'  {c:<22}  dtype={df[c].dtype}  '
              f'sample={df[c].iloc[0]}')

print('\n' + '─'*60)
print('UNSTRUCTURED COLUMNS — free text requiring NLP')
print('─'*60)
for c in ['title','content','name']:
    if c in df.columns:
        print(f'  {c:<22}  e.g. "{str(df[c].iloc[0])[:55]}..."')

print('''
COMPARISON:
┌──────────────┬──────────────────────┬───────────────────────────┐
│ Property     │ Structured           │ Unstructured              │
├──────────────┼──────────────────────┼───────────────────────────┤
│ Format       │ Rows / columns       │ Free text, HTML, audio    │
│ Query        │ SQL / aggregations   │ NLP, regex, embeddings    │
│ Volume       │ ~20% of world data   │ ~80% of world data        │
│ Example      │ document_date        │ content (speech text)     │
└──────────────┴──────────────────────┴───────────────────────────┘

BUSINESS APPLICATIONS:
  Marketing  → Identify resonant language for campaign targeting
  Finance    → Track economic policy signals across years
  Healthcare → Monitor reform rhetoric and policy commitments
  Education  → Map investment promises by term / year
''')

# ── Plot ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0f0f1a')

tc = df_raw['document_type_name'].value_counts().head(8)
axes[0].barh(range(len(tc)), tc.values, color=PALETTE[:len(tc)])
axes[0].set_yticks(range(len(tc)))
axes[0].set_yticklabels(tc.index, fontsize=8)
axes[0].invert_yaxis()
axes[0].set_title('Document Types in Full Dataset', fontweight='bold')
axes[0].set_xlabel('Count')

yc = df['year'].value_counts().sort_index().dropna()
axes[1].bar(yc.index.astype(int), yc.values,
            color=PALETTE[1], edgecolor='#0f0f1a', width=0.7)
axes[1].set_title('Development Speeches by Year', fontweight='bold')
axes[1].set_xlabel('Year'); axes[1].set_ylabel('Count')
for bar in axes[1].patches:
    axes[1].text(bar.get_x()+bar.get_width()/2,
                 bar.get_height()+0.1, int(bar.get_height()),
                 ha='center', fontsize=8)

plt.suptitle('Dataset Overview — Structured Data View',
             fontsize=13, color='white', fontweight='bold')
plt.tight_layout()
save_plot('01_dataset_overview.png')

────────────────────────────────────────────────────────────
STRUCTURED COLUMNS — numeric / date / categorical
────────────────────────────────────────────────────────────
  number                  dtype=int64  sample=44
  start_date              dtype=object  sample=2009-01-20 00:00:00
  end_date                dtype=object  sample=2017-01-20 00:00:00
  document_date           dtype=datetime64[ns]  sample=2016-03-23 00:00:00
  year                    dtype=int32  sample=2016

────────────────────────────────────────────────────────────
UNSTRUCTURED COLUMNS — free text requiring NLP
────────────────────────────────────────────────────────────
  title                   e.g. "FACT SHEET:  United States – Argentina Relationship..."
  content                 e.g. "President Obama, accompanied by First Lady Michelle Oba..."
  name                    e.g. "Barack Obama..."

COMPARISON:
┌──────────────┬──────────────────────┬───────────────────────────┐
│ Property     │ Structured           │

---
## 🔤 Text Preprocessing
> **Syllabus:** Tokenization · Stop Words · Stemming · Lemmatization · NLP Challenges

In [8]:
# ══════════════════════════════════════════════════════
# CELL 6 ▸ TOKENIZATION
# ══════════════════════════════════════════════════════
SAMPLE_TEXT  = str(df['content'].iloc[0])[:2000]
SAMPLE_TITLE = str(df['title'].iloc[0])

print(f'Speech: "{SAMPLE_TITLE[:70]}"')
print(f'\nRAW TEXT (first 300 chars):\n  {SAMPLE_TEXT[:300]}\n')

word_tokens = word_tokenize(SAMPLE_TEXT)
sent_tokens = sent_tokenize(SAMPLE_TEXT)

print('─'*60)
print('WORD TOKENIZATION RESULTS:')
print('─'*60)
print(f'  Total word tokens    : {len(word_tokens)}')
print(f'  Unique word tokens   : {len(set(word_tokens))}')
print(f'  First 25 tokens      : {word_tokens[:25]}')

print('\n' + '─'*60)
print('SENTENCE TOKENIZATION RESULTS:')
print('─'*60)
print(f'  Total sentences      : {len(sent_tokens)}')
print(f'  Avg words/sentence   : {len(word_tokens)/max(len(sent_tokens),1):.1f}')
print(f'\n  First 3 sentences:')
for i,s in enumerate(sent_tokens[:3], 1):
    print(f'  [{i}] {s[:120]}')

# ── Plot: Token frequency ──
alpha_tokens = [t.lower() for t in word_tokens if t.isalpha()]
freq = Counter(alpha_tokens)
top30 = freq.most_common(30)
words30, counts30 = zip(*top30)

fig, ax = plt.subplots(figsize=(14, 5))
fig.patch.set_facecolor('#0f0f1a')
bars = ax.bar(range(len(words30)), counts30,
              color=[PALETTE[i%len(PALETTE)] for i in range(30)])
ax.set_xticks(range(len(words30)))
ax.set_xticklabels(words30, rotation=45, ha='right', fontsize=9)
ax.set_title('Top 30 Word Tokens (before stop word removal)',
             fontweight='bold')
ax.set_ylabel('Frequency')
for bar in bars:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
            int(bar.get_height()), ha='center', fontsize=7)
plt.tight_layout()
save_plot('02_token_frequency.png')

Speech: "FACT SHEET:  United States – Argentina Relationship"

RAW TEXT (first 300 chars):
  President Obama, accompanied by First Lady Michelle Obama, is in Argentina to meet with newly elected President Mauricio Macri and First Lady Juliana Awada. Together, the leaders explored opportunities to strengthen the relationship between the United States and Argentina and partner to address glob

────────────────────────────────────────────────────────────
WORD TOKENIZATION RESULTS:
────────────────────────────────────────────────────────────
  Total word tokens    : 317
  Unique word tokens   : 174
  First 25 tokens      : ['President', 'Obama', ',', 'accompanied', 'by', 'First', 'Lady', 'Michelle', 'Obama', ',', 'is', 'in', 'Argentina', 'to', 'meet', 'with', 'newly', 'elected', 'President', 'Mauricio', 'Macri', 'and', 'First', 'Lady', 'Juliana']

────────────────────────────────────────────────────────────
SENTENCE TOKENIZATION RESULTS:
────────────────────────────────────────────────────

In [9]:
# ══════════════════════════════════════════════════════
# CELL 7 ▸ STOP WORD REMOVAL
# ══════════════════════════════════════════════════════
ENGLISH_SW = set(nltk_sw.words('english'))
CUSTOM_SW  = {
    'applause','laughter','audience','mr','ms','mrs','also',
    'would','could','said','let','like','one','two','shall',
    'must','tonight','today','know','get','even','back','still'
}
ALL_SW = ENGLISH_SW | CUSTOM_SW

tokens_lower  = [t.lower() for t in word_tokens if t.isalpha()]
tokens_clean  = [t for t in tokens_lower if t not in ALL_SW]
removed_sw    = [t for t in tokens_lower if t in ALL_SW]
removed_uniq  = sorted(set(removed_sw))

print('STOP WORD REMOVAL RESULTS:')
print(f'  Tokens before removal   : {len(tokens_lower)}')
print(f'  Tokens after removal    : {len(tokens_clean)}')
print(f'  Stop words removed      : {len(removed_sw)}')
print(f'  Unique stops removed    : {len(removed_uniq)}')
print(f'  Retention rate          : {len(tokens_clean)/max(len(tokens_lower),1)*100:.1f}%')
print(f'\n  All unique stop words removed from this speech:')
print(f'  {removed_uniq}')

print(f'\n  Top 15 stop words by frequency in this speech:')
sw_freq = Counter(removed_sw)
for sw, cnt in sw_freq.most_common(15):
    bar = '█' * cnt
    print(f'    {sw:<15} {bar[:40]} ({cnt})')

print(f'\n  Top 20 content words AFTER removal:')
content_freq = Counter(tokens_clean)
for w, c in content_freq.most_common(20):
    print(f'    {w:<20} {c}')

# ── Plot: Before vs After ──
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.patch.set_facecolor('#0f0f1a')

top_before = Counter(tokens_lower).most_common(15)
top_after  = content_freq.most_common(15)

for ax, data, title, color in [
    (axes[0], top_before, 'Before Stop Word Removal', PALETTE[1]),
    (axes[1], top_after,  'After Stop Word Removal',  PALETTE[0])
]:
    w, c = zip(*data)
    ax.barh(range(len(w)), c, color=color)
    ax.set_yticks(range(len(w)))
    ax.set_yticklabels(w, fontsize=9)
    ax.invert_yaxis()
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Frequency')

plt.suptitle('Impact of Stop Word Removal on Token Frequencies',
             fontsize=13, color='white', fontweight='bold')
plt.tight_layout()
save_plot('03_stopword_removal.png')

STOP WORD REMOVAL RESULTS:
  Tokens before removal   : 270
  Tokens after removal    : 175
  Stop words removed      : 95
  Unique stops removed    : 30
  Retention rate          : 64.8%

  All unique stop words removed from this speech:
  ['a', 'also', 'and', 'as', 'at', 'between', 'both', 'by', 'for', 'further', 'has', 'in', 'is', 'it', 'more', 'of', 's', 'such', 'than', 'that', 'the', 'their', 'this', 'those', 'to', 'two', 'was', 'which', 'will', 'with']

  Top 15 stop words by frequency in this speech:
    and             ████████████████████ (20)
    the             █████████████████ (17)
    to              █████████ (9)
    in              ████████ (8)
    of              ██████ (6)
    s               ███ (3)
    will            ███ (3)
    by              ██ (2)
    with            ██ (2)
    a               ██ (2)
    two             ██ (2)
    for             ██ (2)
    both            ██ (2)
    is              █ (1)
    between         █ (1)

  Top 20 content words AFTER r

In [10]:
# ══════════════════════════════════════════════════════
# CELL 8 ▸ STEMMING & LEMMATIZATION
# ══════════════════════════════════════════════════════
stemmer  = PorterStemmer()
lemma    = WordNetLemmatizer()

stems  = [stemmer.stem(t)        for t in tokens_clean]
lemmas = [lemma.lemmatize(t, pos='v') for t in tokens_clean]

print('STEMMING vs LEMMATIZATION — SIDE BY SIDE:')
print(f'  {"Original":<18} {"Stemmed":<18} {"Lemmatized":<18} {"Same?"}')
print('  ' + '─'*68)
seen = set()
for orig, stem, lem in zip(tokens_clean, stems, lemmas):
    if orig not in seen and len(orig) > 5:
        diff = '⚠ differ' if stem != lem else ''
        print(f'  {orig:<18} {stem:<18} {lem:<18} {diff}')
        seen.add(orig)
    if len(seen) >= 25:
        break

print(f'\nVOCABULARY SIZE COMPARISON:')
print(f'  Original tokens : {len(set(tokens_clean))}')
print(f'  After stemming  : {len(set(stems))} '
      f'(reduction: {(1-len(set(stems))/max(len(set(tokens_clean)),1))*100:.1f}%)')
print(f'  After lemmatize : {len(set(lemmas))} '
      f'(reduction: {(1-len(set(lemmas))/max(len(set(tokens_clean)),1))*100:.1f}%)')

print('''
WHY LEMMATIZATION IS PREFERRED FOR POLITICAL SPEECH ANALYSIS:
  • Stemming strips suffixes blindly:  "economies" → "economi"  ❌
  • Lemmatization uses a dictionary:   "economies" → "economy"  ✅
  • Stemming on "investing" → "invest", "universal" → "univers" ❌
  • Lemmatization preserves real words for human-readable output ✅
  • POS-aware lemmatization handles verbs vs nouns correctly     ✅
''')

# ── Plot ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#0f0f1a')

for ax, tokens, title, color in [
    (axes[0], tokens_clean, 'Original Tokens',  PALETTE[2]),
    (axes[1], stems,        'After Stemming',    PALETTE[3]),
    (axes[2], lemmas,       'After Lemmatization', PALETTE[0])
]:
    top = Counter(tokens).most_common(12)
    w, c = zip(*top)
    ax.barh(range(len(w)), c, color=color)
    ax.set_yticks(range(len(w)))
    ax.set_yticklabels(w, fontsize=9)
    ax.invert_yaxis()
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Frequency')
    ax.text(0.98, 0.02, f'Vocab: {len(set(tokens))}',
            transform=ax.transAxes, ha='right',
            fontsize=9, color='#ffd740')

plt.suptitle('Stemming vs Lemmatization — Vocabulary Comparison',
             fontsize=13, color='white', fontweight='bold')
plt.tight_layout()
save_plot('04_stem_vs_lemma.png')

# ── Build final clean corpus ──
def preprocess(text, min_len=3):
    tokens = word_tokenize(str(text).lower())
    tokens = [t for t in tokens if t.isalpha() and t not in ALL_SW and len(t)>=min_len]
    tokens = [lemma.lemmatize(t, pos='v') for t in tokens]
    return tokens

print('\nBuilding clean corpus (all speeches)...')
df['tokens']      = df['content'].apply(preprocess)
df['clean_text']  = df['tokens'].apply(lambda t: ' '.join(t))
df['token_count'] = df['tokens'].apply(len)

print(f'  ✅  Corpus built: {len(df)} speeches')
print(f'  Avg tokens/speech : {df["token_count"].mean():.0f}')
print(f'  Min / Max tokens  : {df["token_count"].min()} / {df["token_count"].max()}')

STEMMING vs LEMMATIZATION — SIDE BY SIDE:
  Original           Stemmed            Lemmatized         Same?
  ────────────────────────────────────────────────────────────────────
  president          presid             president          ⚠ differ
  accompanied        accompani          accompany          ⚠ differ
  michelle           michel             michelle           ⚠ differ
  argentina          argentina          argentina          
  elected            elect              elect              
  mauricio           mauricio           mauricio           
  juliana            juliana            juliana            
  together           togeth             together           ⚠ differ
  leaders            leader             leaders            ⚠ differ
  explored           explor             explore            ⚠ differ
  opportunities      opportun           opportunities      ⚠ differ
  strengthen         strengthen         strengthen         
  relationship       relationship       relati

---
## 🗂️ Bag of Words & TF-IDF
> **Syllabus:** BoW · TF-IDF · N-grams · Applications

In [11]:
# ══════════════════════════════════════════════════════
# CELL 9 ▸ BAG OF WORDS (BoW)
# ══════════════════════════════════════════════════════
corpus = df['clean_text'].tolist()
titles = df['title'].tolist()

bow_vec = CountVectorizer(max_features=500, min_df=2)
bow_mat = bow_vec.fit_transform(corpus)
bow_feat = bow_vec.get_feature_names_out()

print('BAG OF WORDS (BoW) RESULTS:')
print(f'  Vocabulary size    : {len(bow_feat):,}')
print(f'  Matrix shape       : {bow_mat.shape}  (speeches × terms)')
print(f'  Matrix sparsity    : {(1 - bow_mat.nnz / (bow_mat.shape[0]*bow_mat.shape[1]))*100:.1f}%')

bow_dense = bow_mat.toarray()
corpus_freq = bow_dense.sum(axis=0)
top_idx = corpus_freq.argsort()[-20:][::-1]

print(f'\n  Top 20 terms by total corpus frequency (BoW):')
print(f'  {"Term":<22} {"Total Count":>12}')
print('  ' + '─'*36)
for i in top_idx:
    print(f'  {bow_feat[i]:<22} {int(corpus_freq[i]):>12}')

# ── Plot: BoW top terms ──
fig, ax = plt.subplots(figsize=(13, 6))
fig.patch.set_facecolor('#0f0f1a')
top_terms  = [bow_feat[i] for i in top_idx]
top_counts = [corpus_freq[i] for i in top_idx]
bars = ax.barh(range(len(top_terms)), top_counts,
               color=[PALETTE[i%len(PALETTE)] for i in range(len(top_terms))])
ax.set_yticks(range(len(top_terms)))
ax.set_yticklabels(top_terms, fontsize=10)
ax.invert_yaxis()
for bar in bars:
    ax.text(bar.get_width()+1, bar.get_y()+bar.get_height()/2,
            int(bar.get_width()), va='center', fontsize=8)
ax.set_title('Top 20 Terms — Bag of Words (Development Speeches)',
             fontweight='bold')
ax.set_xlabel('Total Frequency Across All Speeches')
plt.tight_layout()
save_plot('05_bow_top_terms.png')

BAG OF WORDS (BoW) RESULTS:
  Vocabulary size    : 500
  Matrix shape       : (13405, 500)  (speeches × terms)
  Matrix sparsity    : 70.2%

  Top 20 terms by total corpus frequency (BoW):
  Term                    Total Count
  ────────────────────────────────────
  president                    188014
  make                         112156
  think                         97438
  state                         93049
  go                            91332
  people                        83766
  work                          83257
  take                          62953
  unite                         62413
  well                          59176
  want                          57335
  time                          51433
  say                           49754
  see                           49290
  right                         48366
  get                           47466
  need                          44390
  come                          44052
  new                           43913
  country   

In [12]:
# ══════════════════════════════════════════════════════
# CELL 10 ▸ TF-IDF VECTORIZATION
# ══════════════════════════════════════════════════════
tfidf_vec = TfidfVectorizer(max_features=500, min_df=2, sublinear_tf=True)
tfidf_mat = tfidf_vec.fit_transform(corpus)
tfidf_feat = tfidf_vec.get_feature_names_out()

tfidf_dense = tfidf_mat.toarray()
mean_tfidf  = tfidf_dense.mean(axis=0)
top_tfidf   = mean_tfidf.argsort()[-20:][::-1]

print('TF-IDF RESULTS:')
print(f'  Formula: TF-IDF(t,d) = TF(t,d) × log(N / df(t))')
print(f'  Vocabulary size  : {len(tfidf_feat):,}')
print(f'  Matrix shape     : {tfidf_mat.shape}')
print(f'  sublinear_tf=True → log(TF) applied to reduce dominance of very frequent terms')

print(f'\n  Top 20 terms by mean TF-IDF score across corpus:')
print(f'  {"Term":<22} {"Mean TF-IDF":>12}')
print('  ' + '─'*36)
for i in top_tfidf:
    print(f'  {tfidf_feat[i]:<22} {mean_tfidf[i]:>12.4f}')

print(f'\n  Per-speech top TF-IDF terms (signature terms):')
print(f'  {"Speech":<50} {"Top Signature Terms"}')
print('  ' + '─'*80)
for i, title in enumerate(titles[:8]):
    row  = tfidf_dense[i]
    tidx = row.argsort()[-5:][::-1]
    terms = [f"{tfidf_feat[j]}({row[j]:.3f})" for j in tidx if row[j]>0]
    print(f'  {title[:49]:<50} {", ".join(terms[:4])}')

# ── Heatmap ──
top20_idx = mean_tfidf.argsort()[-20:][::-1]
subset    = tfidf_dense[:min(12, len(df)), :][:, top20_idx]
top20_t   = tfidf_feat[top20_idx]

fig, ax = plt.subplots(figsize=(18, max(5, min(12,len(df))*0.55)))
fig.patch.set_facecolor('#0f0f1a')
sns.heatmap(subset,
            xticklabels=top20_t,
            yticklabels=[t[:35] for t in titles[:min(12,len(df))]],
            cmap='magma', linewidths=0.3, ax=ax,
            cbar_kws={'shrink':0.8})
ax.set_title('TF-IDF Heatmap — Signature Terms per Speech',
             fontweight='bold', fontsize=13)
ax.set_xlabel('Terms'); ax.set_ylabel('Speeches')
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.tight_layout()
save_plot('06_tfidf_heatmap.png')

TF-IDF RESULTS:
  Formula: TF-IDF(t,d) = TF(t,d) × log(N / df(t))
  Vocabulary size  : 500
  Matrix shape     : (13405, 500)
  sublinear_tf=True → log(TF) applied to reduce dominance of very frequent terms

  Top 20 terms by mean TF-IDF score across corpus:
  Term                    Mean TF-IDF
  ────────────────────────────────────
  president                    0.0709
  state                        0.0637
  unite                        0.0562
  work                         0.0547
  make                         0.0530
  people                       0.0504
  obama                        0.0459
  new                          0.0440
  national                     0.0426
  take                         0.0418
  america                      0.0417
  american                     0.0410
  go                           0.0410
  support                      0.0408
  security                     0.0402
  include                      0.0399
  help                         0.0398
  time             

In [13]:
# ══════════════════════════════════════════════════════
# CELL 11 ▸ N-GRAMS (Unigrams, Bigrams, Trigrams)
# ══════════════════════════════════════════════════════
print('N-GRAM ANALYSIS — Unigrams, Bigrams, Trigrams\n')
raw_corpus = df['content'].str.lower().tolist()  # Use raw text for ngrams

results = {}
for label, ngram, n in [('Unigrams  (n=1)', (1,1), 1),
                         ('Bigrams   (n=2)', (2,2), 2),
                         ('Trigrams  (n=3)', (3,3), 3)]:
    vec = TfidfVectorizer(ngram_range=ngram, max_features=200,
                          min_df=2, stop_words='english')
    mat = vec.fit_transform(raw_corpus)
    feat = vec.get_feature_names_out()
    mean_sc = mat.toarray().mean(axis=0)
    top_idx = mean_sc.argsort()[-15:][::-1]
    results[label] = [(feat[i], mean_sc[i]) for i in top_idx if mean_sc[i]>0]

    print(f'  ── {label} ──')
    print(f'  {"Term":<35} {"Mean TF-IDF":>12}')
    print('  ' + '─'*50)
    for term, score in results[label][:12]:
        print(f'  {term:<35} {score:>12.4f}')
    print()

print('KEY INSIGHT:')
print('  Unigrams miss compound political concepts.')
print('  Bigrams reveal: "middle class", "clean energy", "small business"')
print('  Trigrams reveal: "middle class families", "create new jobs"')

# ── Plot ──
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.patch.set_facecolor('#0f0f1a')

for ax, (label, data), color in zip(
        axes, results.items(), [PALETTE[1], PALETTE[0], PALETTE[2]]):
    if not data: continue
    terms, scores = zip(*data[:12])
    ax.barh(range(len(terms)), scores, color=color)
    ax.set_yticks(range(len(terms)))
    ax.set_yticklabels(terms, fontsize=9)
    ax.invert_yaxis()
    ax.set_title(label.strip(), fontweight='bold')
    ax.set_xlabel('Mean TF-IDF')

plt.suptitle('N-Gram Analysis — Unigrams vs Bigrams vs Trigrams',
             fontsize=13, color='white', fontweight='bold')
plt.tight_layout()
save_plot('07_ngrams.png')

N-GRAM ANALYSIS — Unigrams, Bigrams, Trigrams

  ── Unigrams  (n=1) ──
  Term                                 Mean TF-IDF
  ──────────────────────────────────────────────────
  president                                 0.1366
  united                                    0.0829
  states                                    0.0812
  people                                    0.0761
  applause                                  0.0707
  mr                                        0.0704
  ve                                        0.0681
  going                                     0.0613
  just                                      0.0593
  national                                  0.0544
  new                                       0.0532
  think                                     0.0525

  ── Bigrams   (n=2) ──
  Term                                 Mean TF-IDF
  ──────────────────────────────────────────────────
  united states                             0.1644
  president obama                

In [14]:
# ══════════════════════════════════════════════════════
# CELL 12 ▸ WORD EMBEDDINGS — Word2Vec & GloVe-style
# ══════════════════════════════════════════════════════
print('WORD EMBEDDINGS — Word2Vec\n')
token_lists = df['tokens'].tolist()

w2v_model = Word2Vec(
    sentences   = token_lists,
    vector_size = 100,
    window      = 5,
    min_count   = 2,
    workers     = 2,
    epochs      = 20,
    sg          = 0  # CBOW (sg=1 for Skip-gram)
)

print(f'  Model trained on {len(token_lists)} speeches')
print(f'  Vocabulary size  : {len(w2v_model.wv):,} terms')
print(f'  Vector dimensions: {w2v_model.vector_size}')
print(f'  Window size      : {w2v_model.window}')
print(f'  Algorithm        : CBOW (Continuous Bag of Words)')

QUERY_WORDS = ['economy','education','jobs','energy','healthcare']
for qw in QUERY_WORDS:
    if qw in w2v_model.wv:
        similar = w2v_model.wv.most_similar(qw, topn=6)
        sim_str = ', '.join([f'{w}({s:.2f})' for w,s in similar])
        print(f'\n  Words most similar to "{qw}":')
        print(f'  {sim_str}')

print('''
Word2Vec vs GloVe (conceptual comparison):
┌────────────────┬──────────────────────────┬──────────────────────────┐
│ Property       │ Word2Vec                 │ GloVe                    │
├────────────────┼──────────────────────────┼──────────────────────────┤
│ Approach       │ Predictive (neural net)  │ Count-based (co-occur.)  │
│ Training       │ CBOW or Skip-gram        │ Global co-occurrence mat │
│ Best for       │ Local context patterns   │ Global semantic meaning  │
│ Available here │ ✅ Trained on our corpus  │ Pre-trained (download)   │
└────────────────┴──────────────────────────┴──────────────────────────┘
''')

# ── t-SNE plot of word vectors ──
words_to_plot = [
    'economy','jobs','education','energy','healthcare',
    'invest','growth','infrastructure','innovation',
    'america','people','work','future','tax','budget'
]
words_to_plot = [w for w in words_to_plot if w in w2v_model.wv]
vectors = np.array([w2v_model.wv[w] for w in words_to_plot])

if len(vectors) >= 4:
    perp = min(5, len(vectors)-1)
    tsne = TSNE(n_components=2, perplexity=perp,
                random_state=42, n_iter=1000)
    coords = tsne.fit_transform(vectors)

    fig, ax = plt.subplots(figsize=(10, 8))
    fig.patch.set_facecolor('#0f0f1a')
    for i, word in enumerate(words_to_plot):
        c = PALETTE[i % len(PALETTE)]
        ax.scatter(coords[i,0], coords[i,1], color=c, s=120, zorder=2)
        ax.annotate(word, (coords[i,0]+0.5, coords[i,1]+0.5),
                    fontsize=11, color=c)
    ax.set_title('Word2Vec Embeddings — t-SNE Projection',
                 fontweight='bold')
    ax.set_xlabel('t-SNE dim 1'); ax.set_ylabel('t-SNE dim 2')
    plt.tight_layout()
    save_plot('08_word2vec_tsne.png')

WORD EMBEDDINGS — Word2Vec



Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

  Model trained on 13405 speeches
  Vocabulary size  : 48,494 terms
  Vector dimensions: 100
  Window size      : 5
  Algorithm        : CBOW (Continuous Bag of Words)

  Words most similar to "economy":
  economies(0.62), recession(0.59), economic(0.59), growth(0.57), economically(0.55), market(0.52)

  Words most similar to "education":
  educational(0.58), educations(0.55), school(0.51), preschool(0.50), educate(0.50), quality(0.50)

  Words most similar to "energy":
  energies(0.77), electricity(0.65), cookstoves(0.65), limpias(0.63), bedpans(0.60), pigment(0.59)

  Words most similar to "healthcare":
  providers(0.59), medical(0.55), oncologists(0.55), frontline(0.55), psychiatric(0.55), veterinary(0.54)

Word2Vec vs GloVe (conceptual comparison):
┌────────────────┬──────────────────────────┬──────────────────────────┐
│ Property       │ Word2Vec                 │ GloVe                    │
├────────────────┼──────────────────────────┼──────────────────────────┤
│ Approach       │

---
## 🤖 Text Classification
> **Syllabus:** Naive Bayes · SVM · Logistic Regression

In [15]:
# ══════════════════════════════════════════════════════
# CELL 13 ▸ TEXT CLASSIFICATION — NB, SVM, Logistic
# ══════════════════════════════════════════════════════
# Create labels based on document type or year era
def assign_era(year):
    if pd.isna(year): return 'Unknown'
    y = int(year)
    if y <= 2010: return 'Crisis Era'
    elif y <= 2013: return 'Recovery Era'
    else: return 'Legacy Era'

df['era'] = df['year'].apply(assign_era)
era_counts = df['era'].value_counts()

print('TEXT CLASSIFICATION SETUP:')
print(f'  Task: Classify speeches by presidential era')
print(f'  Classes: {dict(era_counts)}')

# Keep classes with enough samples
valid = era_counts[era_counts >= 3].index.tolist()
df_cls = df[df['era'].isin(valid)].reset_index(drop=True)
print(f'  Classes used for training: {valid}')
print(f'  Speeches for training: {len(df_cls)}')

if len(df_cls) < 6 or len(valid) < 2:
    print('\n  ⚠️  Not enough samples per class for full classification.')
    print('  Showing pipeline structure and cross-validation scores instead.\n')

le = LabelEncoder()
y  = le.fit_transform(df_cls['era'])
X_vec = TfidfVectorizer(max_features=500, sublinear_tf=True)
X     = X_vec.fit_transform(df_cls['clean_text'])

MODELS = {
    'Naive Bayes'       : MultinomialNB(),
    'SVM (LinearSVC)'   : LinearSVC(max_iter=2000),
    'Logistic Regression': LogisticRegression(max_iter=2000)
}

print(f'\n  {"Model":<25} {"CV Accuracy (5-fold)":>22} {"Std":>8}')
print('  ' + '─'*58)
cv_results = {}
for name, model in MODELS.items():
    cv = cross_val_score(model, X, y, cv=min(5, len(df_cls)),
                         scoring='accuracy')
    cv_results[name] = (cv.mean(), cv.std())
    print(f'  {name:<25} {cv.mean()*100:>18.1f}%    {cv.std()*100:>5.1f}%')

# Train best model fully and show classification report
if len(df_cls) >= 6:
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )
    best_model = LogisticRegression(max_iter=2000)
    best_model.fit(X_tr, y_tr)
    y_pred = best_model.predict(X_te)
    print(f'\n  Classification Report (Logistic Regression):')
    print(classification_report(y_te, y_pred,
          target_names=le.classes_, zero_division=0))

# ── Plot: Model comparison ──
fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('#0f0f1a')
names  = list(cv_results.keys())
means  = [v[0]*100 for v in cv_results.values()]
stds   = [v[1]*100 for v in cv_results.values()]
bars   = ax.bar(names, means, color=PALETTE[:3],
                edgecolor='#0f0f1a', width=0.5)
ax.errorbar(names, means, yerr=stds, fmt='none',
            color='white', capsize=6, linewidth=2)
for bar, m in zip(bars, means):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
            f'{m:.1f}%', ha='center', fontsize=11, fontweight='bold')
ax.set_ylim(0, 115)
ax.set_title('Text Classification — Model Accuracy Comparison (5-fold CV)',
             fontweight='bold')
ax.set_ylabel('Accuracy (%)')
plt.tight_layout()
save_plot('09_classification_accuracy.png')

TEXT CLASSIFICATION SETUP:
  Task: Classify speeches by presidential era
  Classes: {'Legacy Era': np.int64(5031), 'Recovery Era': np.int64(4786), 'Crisis Era': np.int64(3588)}
  Classes used for training: ['Legacy Era', 'Recovery Era', 'Crisis Era']
  Speeches for training: 13405

  Model                       CV Accuracy (5-fold)      Std
  ──────────────────────────────────────────────────────────
  Naive Bayes                             51.7%      4.1%
  SVM (LinearSVC)                         64.6%      6.7%
  Logistic Regression                     64.8%      6.7%

  Classification Report (Logistic Regression):
              precision    recall  f1-score   support

  Crisis Era       0.75      0.70      0.72       897
  Legacy Era       0.70      0.71      0.70      1258
Recovery Era       0.63      0.65      0.64      1197

    accuracy                           0.69      3352
   macro avg       0.69      0.69      0.69      3352
weighted avg       0.69      0.69      0.69     

---
## 💬 Sentiment Analysis
> **Syllabus:** Rule-Based (VADER) · ML Sentiment · Aspect-Based · Social Media Applications

In [16]:
# ══════════════════════════════════════════════════════
# CELL 14 ▸ RULE-BASED SENTIMENT — VADER  (optimised)
# ══════════════════════════════════════════════════════
vader = SentimentIntensityAnalyzer()

# Augment VADER with political domain lexicon
POLITICAL_LEX = {
    'prosperity':3.0,'innovation':2.5,'investment':1.8,
    'revitalize':2.5,'rebuild':2.0,'strengthen':2.0,
    'crisis':-2.5,'recession':-2.5,'unemployment':-2.2,
    'inequality':-2.0,'stagnation':-2.0,'opportunity':2.5,
    'promise':2.0,'progress':2.2,'future':1.5,'together':1.5
}
vader.lexicon.update(POLITICAL_LEX)
print('RULE-BASED SENTIMENT ANALYSIS (VADER):')
print(f'  Custom political lexicon added: {len(POLITICAL_LEX)} terms')
print(f'  Scoring: neg [-1,0] · neu [0,1] · pos [0,1] · compound [-1,+1]')
print(f'  Classification: compound≥+0.15→Optimistic · ≤-0.05→Urgent · else→Neutral')

# ── FIX 1: Truncate text to 2000 chars — VADER is calibrated
#           for short text; scoring full speeches is slow & inaccurate
# ── FIX 2: Score the whole truncated text once — no paragraph loop
# ── FIX 3: Assign columns directly — avoids pd.concat conflict
def sentiment_score(text):
    sc  = vader.polarity_scores(str(text)[:2000])
    c   = sc['compound']
    cat = 'Optimistic' if c >= 0.15 else ('Urgent' if c <= -0.05 else 'Neutral')
    return pd.Series({
        'compound' : round(c, 4),
        'pos'      : round(sc['pos'], 4),
        'neg'      : round(sc['neg'], 4),
        'neu'      : round(sc['neu'], 4),
        'sentiment': cat
    })

# Drop columns if they already exist (safe re-run)
for col in ['compound','pos','neg','neu','sentiment']:
    if col in df.columns:
        df.drop(columns=[col], inplace=True)

sent_df = df['content'].apply(sentiment_score)
df[['compound','pos','neg','neu','sentiment']] = sent_df

print(f'\n  {"Speech":<52} {"Compound":>9}  {"Sentiment"}')
print('  ' + '─'*75)
for _, row in df.iterrows():
    icon = '🟢' if row['sentiment']=='Optimistic' else ('🔴' if row['sentiment']=='Urgent' else '🟡')
    print(f'  {str(row["title"])[:51]:<52} {row["compound"]:>+9.4f}  {icon} {row["sentiment"]}')

print(f'\n  SENTIMENT DISTRIBUTION:')
for cat, cnt in df['sentiment'].value_counts().items():
    pct = cnt/len(df)*100
    bar = '█'*int(pct/5)
    print(f'    {cat:<12} {bar:<20} {cnt} speeches ({pct:.0f}%)')

print(f'\n  Average compound score  : {df["compound"].mean():+.4f}')
print(f'  Most optimistic speech  : {df.loc[df["compound"].idxmax(),"title"][:60]}')
print(f'  Most urgent speech      : {df.loc[df["compound"].idxmin(),"title"][:60]}')

# ── Plots ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#0f0f1a')

# 1. Sentiment distribution pie
scounts    = df['sentiment'].value_counts()
colors_pie = {'Optimistic':'#2ecc71','Urgent':'#e74c3c','Neutral':'#f39c12'}
axes[0].pie(scounts.values,
            labels=scounts.index,
            colors=[colors_pie.get(c,'#888') for c in scounts.index],
            autopct='%1.0f%%', startangle=90,
            textprops={'color':'white','fontsize':11})
axes[0].set_title('Sentiment Distribution', fontweight='bold')

# 2. Compound score per speech
colors_bar = [colors_pie.get(s,'#888') for s in df['sentiment']]
axes[1].bar(range(len(df)), df['compound'], color=colors_bar,
            edgecolor='#0f0f1a', width=0.7)
axes[1].axhline(0.15,  color='#2ecc71', linestyle='--', alpha=0.6, label='+0.15 Optimistic')
axes[1].axhline(-0.05, color='#e74c3c', linestyle='--', alpha=0.6, label='-0.05 Urgent')
axes[1].axhline(0,     color='white',   linestyle='-',  alpha=0.2)
axes[1].set_title('VADER Compound Score per Speech', fontweight='bold')
axes[1].set_xlabel('Speech Index'); axes[1].set_ylabel('Compound')
axes[1].legend(fontsize=8)

# 3. Sentiment arc over years
year_sent = df.groupby('year')['compound'].mean().dropna()
axes[2].plot(year_sent.index.astype(int), year_sent.values,
             color=PALETTE[0], linewidth=2.5, marker='o', markersize=8)
axes[2].fill_between(year_sent.index.astype(int), year_sent.values,
                     alpha=0.2, color=PALETTE[0])
axes[2].axhline(0, color='white', alpha=0.2)
axes[2].set_title('Avg Sentiment Arc by Year', fontweight='bold')
axes[2].set_xlabel('Year'); axes[2].set_ylabel('Mean Compound')

plt.suptitle('VADER Sentiment Analysis — Obama Development Speeches',
             fontsize=13, color='white', fontweight='bold')
plt.tight_layout()
save_plot('10_sentiment_analysis.png')


RULE-BASED SENTIMENT ANALYSIS (VADER):
  Custom political lexicon added: 16 terms
  Scoring: neg [-1,0] · neu [0,1] · pos [0,1] · compound [-1,+1]
  Classification: compound≥+0.15→Optimistic · ≤-0.05→Urgent · else→Neutral

  Speech                                                Compound  Sentiment
  ───────────────────────────────────────────────────────────────────────────
  FACT SHEET:  United States – Argentina Relationship    +0.9969  🟢 Optimistic
  FACT SHEET: U.S.-China Economic Relations              +0.9966  🟢 Optimistic
  Presidential Memorandum -- Establishing Policies fo    -0.9961  🔴 Urgent
  Weekly Address: Reducing Carbon Pollution in Our Po    +0.9621  🟢 Optimistic
  Presidential Memorandum -- Accelerating Technology     +0.9961  🟢 Optimistic
  Weekly Address: Paying Tribute to our Fallen Heroes    +0.9887  🟢 Optimistic
  Weekly Address: Working When Congress Won’t Act        +0.9922  🟢 Optimistic
  Weekly Address: The First Lady Marks Mother’s Day a    +0.9569  🟢 Optimi

In [19]:
# ══════════════════════════════════════════════════════
# CELL 15 ▸ ASPECT-BASED SENTIMENT ANALYSIS (OPTIMIZED)
# ══════════════════════════════════════════════════════

# ── GUARD ──
if 'df' not in dir() or df is None:
    raise RuntimeError(
        '\n\n❌ df is not defined.\n'
        'Run previous cells first.\n'
    )
if 'vader' not in dir():
    raise RuntimeError('\n\n❌ vader not defined.\n')
if 'sent_tokenize' not in dir():
    raise RuntimeError('\n\n❌ sent_tokenize not imported.\n')

print('ASPECT-BASED SENTIMENT ANALYSIS:')
print('Scoring sentiment separately for each policy domain\n')

ASPECTS = {
    'Economy'       : ['economy','economic','gdp','recession','recovery','growth','fiscal'],
    'Jobs'          : ['job','employ','unemployment','worker','workforce','hire','wages'],
    'Education'     : ['education','school','college','university','student','teacher','learn'],
    'Healthcare'    : ['health','healthcare','insurance','medical','hospital','patient','care'],
    'Energy'        : ['energy','oil','solar','wind','clean','renewable','environment','climate'],
    'Infrastructure': ['infrastructure','road','bridge','broadband','transport','build','repair']
}

# ── OPTIMIZATION: Precompute sentences + sentiment ONCE ──
df['sentences'] = df['content'].apply(lambda t: sent_tokenize(str(t)[:1500]))

df['sentence_scores'] = df['sentences'].apply(
    lambda sents: [(sent, vader.polarity_scores(sent)['compound']) for sent in sents]
)

def aspect_sentiment_fast(sentence_scores, keywords):
    scores = [
        score for sent, score in sentence_scores
        if any(kw in sent.lower() for kw in keywords)
    ]
    return round(sum(scores)/len(scores), 4) if scores else 0.0

# ── Compute aspect scores ──
print(f'  {"Aspect":<16}', end='')
for t in df['title'].iloc[:6]:
    print(f' {str(t)[:14]:>15}', end='')
print(f' {"MEAN":>8}')
print('  ' + '─'*120)

aspect_matrix = {}

for aspect, kws in ASPECTS.items():
    scores = df['sentence_scores'].apply(lambda s: aspect_sentiment_fast(s, kws))
    aspect_matrix[aspect] = scores.values
    mean_sc = scores.mean()

    print(f'  {aspect:<16}', end='')
    for sc in scores.iloc[:6]:
        icon = '+' if sc > 0.05 else ('-' if sc < -0.05 else '~')
        print(f' {icon}{sc:>13.3f}', end='')
    print(f' {mean_sc:>+8.3f}')

# ── Heatmap (MEMORY SAFE) ──
aspect_df = pd.DataFrame(aspect_matrix, index=df['title'].str[:30])

# LIMIT rows to avoid huge memory usage
aspect_df = aspect_df.head(25)

fig, ax = plt.subplots(figsize=(10, 6))   # fixed small size
fig.patch.set_facecolor('#0f0f1a')

sns.heatmap(
    aspect_df,
    cmap='RdYlGn',
    center=0,
    linewidths=0.3,
    ax=ax,
    annot=False,   # ❌ removed annotations (major fix)
    cbar_kws={'shrink':0.7}
)

ax.set_title('Aspect-Based Sentiment — Policy Domains × Speeches',
             fontweight='bold', fontsize=12)
ax.set_xlabel('Policy Domain')
ax.set_ylabel('Speech')

plt.xticks(rotation=30, ha='right')
plt.tight_layout()

# ✅ SAME saving function (your folder stays same)
save_plot('11_aspect_sentiment.png')

ASPECT-BASED SENTIMENT ANALYSIS:
Scoring sentiment separately for each policy domain

  Aspect            FACT SHEET:  U  FACT SHEET: U.  Presidential M  Weekly Address  Presidential M  Weekly Address     MEAN
  ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  Economy          +        0.411 +        0.373 -       -0.637 +        0.674 +        0.758 ~        0.000   +0.162
  Jobs             ~        0.000 +        0.625 -       -0.765 ~        0.000 +        0.802 ~        0.000   +0.119
  Education        ~        0.000 ~        0.000 ~        0.000 ~        0.000 ~        0.000 ~        0.000   +0.103
  Healthcare       ~        0.000 ~        0.000 -       -0.637 ~       -0.046 ~        0.000 ~        0.000   +0.115
  Energy           +        0.929 ~        0.000 ~        0.000 +        0.083 ~        0.000 ~        0.000   +0.102
  Infrastructure   +        0.574 +        0.914 +        0.202 ~       -0.03

---
## 🗺️ Topic Modelling
> **Syllabus:** LDA · LSA · K-Means Clustering · Hierarchical Clustering

In [23]:
# ══════════════════════════════════════════════════════
# CELL 16 ▸ LDA — LATENT DIRICHLET ALLOCATION
# ══════════════════════════════════════════════════════
print('TOPIC MODELLING — Latent Dirichlet Allocation (LDA)\n')
print('Theory: LDA assumes each document is a mixture of K topics.')
print('Each topic is a probability distribution over vocabulary words.\n')

NOISE = {
    'also','must','will','shall','every','would','could',
    'said','say','just','like','know','get','put','got',
    'one','two','new','back','now','time','year','today'
}

token_lists = [
    [t for t in toks if t not in NOISE and len(t)>3]
    for toks in df['tokens']
]

id2word = corpora.Dictionary(token_lists)
id2word.filter_extremes(no_below=2, no_above=0.85)
bow_corp = [id2word.doc2bow(doc) for doc in token_lists]

print(f'  Dictionary size : {len(id2word):,} terms')
print(f'  Corpus docs     : {len(bow_corp)}')

NUM_TOPICS = 5
TOPIC_LABELS = {
    0:'Economic Recovery & Jobs',
    1:'Infrastructure & Clean Energy',
    2:'Education & Workforce',
    3:'Healthcare & Social Safety',
    4:'Trade, Innovation & Competitiveness'
}

lda = LdaModel(
    corpus=bow_corp, id2word=id2word,
    num_topics=NUM_TOPICS, random_state=42,
    passes=20, alpha='auto', eta='auto'
)

print(f'\n  LDA trained: K={NUM_TOPICS} topics, 20 passes\n')
print(f'  {"Topic":>5}  {"Label":<40} {"Top 10 Words"}')
print('  ' + '─'*95)

for tid in range(NUM_TOPICS):
    ww    = lda.show_topic(tid, topn=10)
    words = ', '.join([f'{w}({s:.3f})' for w,s in ww])
    label = TOPIC_LABELS[tid]
    print(f'  {tid:>5}  {label:<40} {words}')

# Assign dominant topic
def dominant_topic(bow_doc):
    dist = lda.get_document_topics(bow_doc)
    return max(dist, key=lambda x:x[1])[0] if dist else 0

df['lda_topic']    = [dominant_topic(b) for b in bow_corp]
df['topic_label']  = df['lda_topic'].map(TOPIC_LABELS)

print(f'\n  Speech → Topic Assignment:')
print(f'  {"Speech":<52} {"Topic Label"}')
print('  ' + '─'*80)

for _, row in df.iterrows():
    print(f'  {str(row["title"])[:51]:<52} {row["topic_label"]}')

# ── Plot ──
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#0f0f1a')

# Topic distribution bar
tc = df['topic_label'].value_counts()

axes[0].barh(range(len(tc)), tc.values,
             color=[PALETTE[i%len(PALETTE)] for i in range(len(tc))])

axes[0].set_yticks(range(len(tc)))
axes[0].set_yticklabels(tc.index, fontsize=9)
axes[0].invert_yaxis()
axes[0].set_title('LDA Topic Distribution', fontweight='bold')
axes[0].set_xlabel('Number of Speeches')

for i, v in enumerate(tc.values):
    axes[0].text(v+0.1, i, str(v), va='center', fontsize=9)

# Topic-word heatmap for top 8 words per topic
top_n_words = 8
topic_word_mat = np.zeros((NUM_TOPICS, top_n_words))
all_words = []

for tid in range(NUM_TOPICS):
    ww = lda.show_topic(tid, topn=top_n_words)
    for j, (w, s) in enumerate(ww):
        topic_word_mat[tid, j] = s
        if tid == 0:
            all_words.append(w)

topic_names = [TOPIC_LABELS[i][:20] for i in range(NUM_TOPICS)]
word_labels = [lda.show_topic(0, topn=top_n_words)[j][0] for j in range(top_n_words)]

sns.heatmap(topic_word_mat, xticklabels=word_labels,
            yticklabels=topic_names,
            cmap='YlOrRd', ax=axes[1], linewidths=0.3,
            annot=True, fmt='.3f', cbar_kws={'shrink':0.8})

axes[1].set_title('Top Word Probabilities per Topic', fontweight='bold')

plt.suptitle('LDA Topic Modelling Results',
             fontsize=13, color='white', fontweight='bold')

plt.tight_layout()
save_plot('12_lda_topics.png')

TOPIC MODELLING — Latent Dirichlet Allocation (LDA)

Theory: LDA assumes each document is a mixture of K topics.
Each topic is a probability distribution over vocabulary words.

  Dictionary size : 41,684 terms
  Corpus docs     : 13405

  LDA trained: K=5 topics, 20 passes

  Topic  Label                                    Top 10 Words
  ───────────────────────────────────────────────────────────────────────────────────────────────
      0  Economic Recovery & Jobs                 make(0.016), people(0.014), work(0.012), want(0.011), think(0.008), right(0.008), country(0.007), come(0.007), take(0.007), need(0.006)
      1  Infrastructure & Clean Energy            energy(0.011), state(0.010), support(0.008), work(0.007), unite(0.007), help(0.007), program(0.007), build(0.006), economic(0.005), global(0.005)
      2  Education & Workforce                    state(0.016), unite(0.012), order(0.011), national(0.010), federal(0.010), provide(0.007), include(0.007), health(0.007), governmen

In [24]:
# ══════════════════════════════════════════════════════
# CELL 17 ▸ LSA — LATENT SEMANTIC ANALYSIS
# ══════════════════════════════════════════════════════
print('LATENT SEMANTIC ANALYSIS (LSA):')
print('Method: TF-IDF → SVD (Singular Value Decomposition)')
print('Captures latent semantic structure by decomposing term-doc matrix\n')

n_comp = min(5, len(df)-1)
svd    = TruncatedSVD(n_components=n_comp, random_state=42)
lsa_mat = svd.fit_transform(tfidf_mat)

print(f'  TF-IDF matrix shape   : {tfidf_mat.shape}')
print(f'  LSA components        : {n_comp}')
print(f'  Variance explained    : {svd.explained_variance_ratio_.sum()*100:.1f}%')

print(f'\n  Variance explained per component:')
for i, v in enumerate(svd.explained_variance_ratio_):
    bar = '█' * int(v*200)
    print(f'  Component {i+1}: {bar[:40]} {v*100:.1f}%')

print(f'\n  Top terms per LSA component (latent semantic dimensions):')
for i in range(n_comp):
    top_idx = svd.components_[i].argsort()[-10:][::-1]
    terms   = [tfidf_feat[j] for j in top_idx]
    print(f'  Component {i+1}: {", ".join(terms)}')

# ── Plot: LSA component 1 vs 2 ──
if n_comp >= 2:
    fig, ax = plt.subplots(figsize=(10, 7))
    fig.patch.set_facecolor('#0f0f1a')
    for i, (x, y) in enumerate(zip(lsa_mat[:,0], lsa_mat[:,1])):
        c = PALETTE[df['lda_topic'].iloc[i] % len(PALETTE)]
        ax.scatter(x, y, color=c, s=120, zorder=2)
        ax.annotate(str(df['title'].iloc[i])[:20],
                    (x+0.002, y+0.002), fontsize=8, color=c)
    ax.set_title('LSA — Document Space (Component 1 vs 2)',
                 fontweight='bold')
    ax.set_xlabel('LSA Component 1'); ax.set_ylabel('LSA Component 2')
    plt.tight_layout()
    save_plot('13_lsa_document_space.png')

LATENT SEMANTIC ANALYSIS (LSA):
Method: TF-IDF → SVD (Singular Value Decomposition)
Captures latent semantic structure by decomposing term-doc matrix

  TF-IDF matrix shape   : (13405, 500)
  LSA components        : 5
  Variance explained    : 20.1%

  Variance explained per component:
  Component 1: ████████ 4.5%
  Component 2: ████████████ 6.3%
  Component 3: ███████ 3.7%
  Component 4: ██████ 3.1%
  Component 5: ████ 2.5%

  Top terms per LSA component (latent semantic dimensions):
  Component 1: president, make, state, work, people, go, unite, want, take, think
  Component 2: state, national, unite, university, department, director, president, development, obama, international
  Component 3: school, university, serve, college, america, education, families, service, member, americans
  Component 4: university, member, director, board, serve, vice, senior, secretary, position, committee
  Component 5: unite, minister, prime, president, world, people, state, peace, obama, serve
  💾  S

In [25]:
# ══════════════════════════════════════════════════════
# CELL 18 ▸ K-MEANS CLUSTERING
# ══════════════════════════════════════════════════════
print('K-MEANS CLUSTERING:\n')
print('Algorithm: Iteratively assigns documents to nearest centroid,')
print('then recomputes centroids until convergence.\n')

K = min(4, max(2, len(df)//3))
km = KMeans(n_clusters=K, random_state=42, n_init=10, max_iter=300)
km_labels = km.fit_predict(tfidf_mat.toarray())
df['kmeans_cluster'] = km_labels

print(f'  K (clusters) = {K}')
print(f'  Inertia (within-cluster SS) = {km.inertia_:.2f}')

# Describe each cluster
print(f'\n  Cluster Assignments:')
for k in range(K):
    members = df[df['kmeans_cluster']==k]['title'].tolist()
    print(f'\n  Cluster {k}  ({len(members)} speeches):')
    for m in members:
        print(f'    • {str(m)[:70]}')

# Top terms per cluster centroid
print(f'\n  Top 10 terms per cluster centroid:')
order_centroids = km.cluster_centers_.argsort()[:,::-1]
for k in range(K):
    terms = [tfidf_feat[i] for i in order_centroids[k,:10]]
    print(f'  Cluster {k}: {', '.join(terms)}')

# Elbow method
max_k = min(8, len(df)-1)
inertias = []
k_range  = range(2, max_k+1)
for k in k_range:
    km_tmp = KMeans(n_clusters=k, random_state=42, n_init=10)
    km_tmp.fit(tfidf_mat)
    inertias.append(km_tmp.inertia_)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0f0f1a')

# Elbow plot
axes[0].plot(list(k_range), inertias, color=PALETTE[0],
             linewidth=2.5, marker='o', markersize=8)
axes[0].axvline(K, color=PALETTE[2], linestyle='--', alpha=0.7,
                label=f'Chosen K={K}')
axes[0].set_title('Elbow Method — Optimal K Selection', fontweight='bold')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia (Within-cluster SS)')
axes[0].legend()

# Cluster membership bar
cluster_counts = df['kmeans_cluster'].value_counts().sort_index()
axes[1].bar([f'Cluster {i}' for i in cluster_counts.index],
            cluster_counts.values,
            color=[PALETTE[i%len(PALETTE)] for i in range(len(cluster_counts))])
axes[1].set_title('K-Means Cluster Sizes', fontweight='bold')
axes[1].set_ylabel('Number of Speeches')
for i, v in enumerate(cluster_counts.values):
    axes[1].text(i, v+0.1, str(v), ha='center', fontsize=11, fontweight='bold')

plt.suptitle('K-Means Document Clustering',
             fontsize=13, color='white', fontweight='bold')
plt.tight_layout()
save_plot('14_kmeans_clustering.png')

K-MEANS CLUSTERING:

Algorithm: Iteratively assigns documents to nearest centroid,
then recomputes centroids until convergence.

  K (clusters) = 4
  Inertia (within-cluster SS) = 8793.48

  Cluster Assignments:

  Cluster 0  (1101 speeches):
    • Presidential Memorandum--Arctic Research and Policy Act
    • Presidential Memorandum -- The Director of the Office of Science and T
    • President Obama Announces More Key Administration Posts
    • President Obama Announces More Key Administration Posts
    • President Obama Announces More Key Administration Posts, 4/6/10
    • Presidential Memorandum -- Designation of Officers of the Office of Pe
    • President Obama Announces More Key Administration Posts, 7-28-09
    • President Obama Announces Presidential Delegation to Monrovia, Liberia
    • President Obama Announces More Key Administration Posts, 5-28-09
    • Presidential Memorandum--Designation of Officials to Act as Director o
    • Memorandum from the President on the Office o

In [27]:
# ══════════════════════════════════════════════════════
# CELL 19 ▸ HIERARCHICAL CLUSTERING + DENDROGRAM
# ══════════════════════════════════════════════════════
print('HIERARCHICAL CLUSTERING:\n')
print('Algorithm: Agglomerative — merges nearest document pairs')
print('bottom-up until all documents are in one cluster.\n')
print('Linkage: Ward (minimises within-cluster variance)\n')

# ── LIMIT DATA SIZE (prevents memory error) ──
MAX_DOCS = 60
if len(df) > MAX_DOCS:
    df_hc = df.head(MAX_DOCS)
    tfidf_mat_hc = tfidf_mat[:MAX_DOCS]
else:
    df_hc = df
    tfidf_mat_hc = tfidf_mat

# Use LSA-reduced features for better distance computation
n_lsa = min(10, len(df_hc)-1)
svd_h = TruncatedSVD(n_components=n_lsa, random_state=42)
X_lsa = svd_h.fit_transform(tfidf_mat_hc)

# Agglomerative clustering
n_hclusters = min(4, max(2, len(df_hc)//3))
hc = AgglomerativeClustering(n_clusters=n_hclusters, linkage='ward')
hc_labels = hc.fit_predict(X_lsa)
df_hc['hc_cluster'] = hc_labels

print(f'  LSA dimensions used   : {n_lsa}')
print(f'  Clusters formed       : {n_hclusters}')
print(f'\n  Cluster membership:')

for k in range(n_hclusters):
    members = df_hc[df_hc['hc_cluster']==k]['title'].tolist()
    print(f'\n  HC Cluster {k} ({len(members)} speeches):')
    for m in members:
        print(f'    • {str(m)[:70]}')

# ── Dendrogram (OPTIMIZED) ──
short_labels = [str(t)[:25] for t in df_hc['title'].tolist()]
Z = linkage(X_lsa, method='ward')

fig, ax = plt.subplots(figsize=(10, 6))   # ↓ fixed size (major fix)
fig.patch.set_facecolor('#0f0f1a')

dendrogram(
    Z,
    labels=short_labels,
    orientation='right',
    leaf_font_size=8,                      # ↓ smaller font
    color_threshold=0.7*max(Z[:,2]),
    ax=ax
)

ax.set_title('Hierarchical Clustering Dendrogram — Obama Development Speeches',
             fontweight='bold', fontsize=12)
ax.set_xlabel('Distance (Ward Linkage)')

plt.tight_layout()

# ✅ SAME SAVE LOCATION
save_plot('15_hierarchical_dendrogram.png')

HIERARCHICAL CLUSTERING:

Algorithm: Agglomerative — merges nearest document pairs
bottom-up until all documents are in one cluster.

Linkage: Ward (minimises within-cluster variance)

  LSA dimensions used   : 10
  Clusters formed       : 4

  Cluster membership:

  HC Cluster 0 (14 speeches):
    • FACT SHEET:  United States – Argentina Relationship
    • FACT SHEET: U.S.-China Economic Relations
    • Presidential Memorandum -- Establishing Policies for Addressing Domest
    • Presidential Memorandum -- Accelerating Technology Transfer and Commer
    • FACT SHEET ON SYRIA
    • Weekly Address: Pursuing a Diplomatic Solution in Syria
    • Fact Sheet: Digital Promise Initiative
    • Presidential Memorandum -- Expanding America's Leadership in Wireless 
    • Presidential Memorandum -- Transforming our Nation's Electric Grid Thr
    • FACT SHEET:  United States Military Nuclear Material Security
    • FACT SHEET: Healthy Communities Challenge for Denver
    • United States National P

---
## 🏷️ Named Entity Recognition (NER)
> **Syllabus:** Rule-Based NER · ML-Based NER (spaCy) · Applications

In [32]:
# ══════════════════════════════════════════════════════
# CELL 20 ▸ NAMED ENTITY RECOGNITION (NER) — FAST VERSION
# ══════════════════════════════════════════════════════
import nltk
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('maxent_ne_chunker_tab', quiet=True)

# ✅ FIX: add missing resources
nltk.download('maxent_ne_chunker', quiet=True)
nltk.download('words', quiet=True)

print('NAMED ENTITY RECOGNITION (NER):\n')

TARGET_ENT = {
    'GPE'   :'Geopolitical Entity (Country/City/State)',
    'ORG'   :'Organization (Gov/Corp/Institution)',
    'PERSON':'Person (Leaders/Officials)',
    'NORP'  :'Nationality / Political Group',
    'FAC'   :'Facility / Infrastructure',
    'MONEY' :'Monetary Value (Budget/Investment)'
}

print('RULE-BASED NER (NLTK) — Sample on first speech:')
sample_text = str(df['content'].iloc[0])[:1000]
nltk_tokens = word_tokenize(sample_text)
pos_tagged  = pos_tag(nltk_tokens)
ne_tree     = ne_chunk(pos_tagged, binary=False)

nltk_entities = defaultdict(list)
for subtree in ne_tree:
    if hasattr(subtree, 'label'):
        entity = ' '.join([t for t,p in subtree.leaves()])
        nltk_entities[subtree.label()].append(entity)

print(f'  NLTK entities found:')
for etype, ents in nltk_entities.items():
    print(f'  [{etype}]  {list(set(ents))[:6]}')

print('\n' + '─'*60)
print('ML-BASED NER (spaCy) — Full corpus analysis:')
print('─'*60)

# ── SPEED OPTIMIZATION ──
MAX_DOCS = 40        # limit number of speeches
TEXT_LIMIT = 15000   # reduce text size per speech

df_ner = df.head(MAX_DOCS)

ner_freq = defaultdict(Counter)

# Use spaCy pipe (MUCH faster than loop)
docs = nlp.pipe(
    (str(text)[:TEXT_LIMIT] for text in df_ner['content']),
    batch_size=10
)

for doc in docs:
    for ent in doc.ents:
        if ent.label_ in TARGET_ENT:
            norm = ent.text.replace("'s","").strip().title()
            if len(norm) > 2:
                ner_freq[ent.label_][norm] += 1

for label, desc in TARGET_ENT.items():
    if label not in ner_freq:
        continue
    print(f'\n  [{label}] {desc}')
    print(f'  {"Entity":<32} {"Mentions":>8}')
    print('  ' + '─'*42)
    for ent, cnt in ner_freq[label].most_common(10):
        print(f'  {ent:<32} {cnt:>8}')

# ── Plot ──
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.patch.set_facecolor('#0f0f1a')
axes_flat = axes.flatten()

for ax, (label, desc), color in zip(
        axes_flat, TARGET_ENT.items(), PALETTE):
    if label not in ner_freq:
        ax.set_visible(False); continue
    top = ner_freq[label].most_common(10)
    if not top:
        ax.set_visible(False); continue
    names, counts = zip(*top)
    ax.barh(range(len(names)), counts, color=color)
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=9)
    ax.invert_yaxis()
    ax.set_title(f'[{label}] {desc[:35]}', fontweight='bold', fontsize=9)
    ax.set_xlabel('Mentions')
    for i, c in enumerate(counts):
        ax.text(c+0.1, i, str(c), va='center', fontsize=8)

plt.suptitle('Named Entity Recognition — Obama Development Speeches',
             fontsize=13, color='white', fontweight='bold')
plt.tight_layout()
save_plot('16_ner_analysis.png')

NAMED ENTITY RECOGNITION (NER):

RULE-BASED NER (NLTK) — Sample on first speech:
  NLTK entities found:
  [PERSON]  ['First Lady Juliana Awada', 'Mauricio Macri', 'First Lady Michelle Obama', 'Obama', 'Macri', 'Argentina']
  [GPE]  ['U.S.', 'United States', 'Argentina']
  [ORGANIZATION]  ['Economic Relationship Since', 'Argentina']

────────────────────────────────────────────────────────────
ML-BASED NER (spaCy) — Full corpus analysis:
────────────────────────────────────────────────────────────

  [GPE] Geopolitical Entity (Country/City/State)
  Entity                           Mentions
  ──────────────────────────────────────────
  The United States                     178
  America                                62
  Argentina                              49
  China                                  39
  U.S.                                   37
  Washington                             20
  Syria                                  19
  Wisconsin                              13
  Iraq 

---
## 📝 Text Summarization
> **Syllabus:** Extractive (TextRank) · Abstractive · Applications

In [33]:
# ══════════════════════════════════════════════════════
# CELL 21 ▸ EXTRACTIVE SUMMARIZATION — TextRank
# ══════════════════════════════════════════════════════
print('TEXT SUMMARIZATION\n')
print('EXTRACTIVE (TextRank):')
print('Algorithm:')
print('  1. Segment text into sentences')
print('  2. TF-IDF vectorize each sentence')
print('  3. Build cosine-similarity graph (sentences = nodes)')
print('  4. PageRank to score each sentence by importance')
print('  5. Return top-N sentences in original reading order\n')

def textrank_summarize(text, n_sents=5):
    sents = sent_tokenize(str(text))
    sents = [s for s in sents if len(s.split())>8]
    if len(sents) <= n_sents:
        return sents
    vec  = TfidfVectorizer(stop_words='english', max_features=500)
    mat  = vec.fit_transform(sents)
    sim  = cosine_similarity(mat)
    np.fill_diagonal(sim, 0)
    G    = nx.from_numpy_array(sim)
    scores = nx.pagerank(G, max_iter=200)
    ranked = sorted(scores, key=scores.get, reverse=True)[:n_sents]
    return [sents[i] for i in sorted(ranked)]

# Summarize first 3 speeches
for i in range(min(3, len(df))):
    title   = str(df['title'].iloc[i])
    content = str(df['content'].iloc[i])
    summary = textrank_summarize(content, n_sents=4)
    orig_wc = len(content.split())
    summ_wc = sum(len(s.split()) for s in summary)
    comp    = (1 - summ_wc/max(orig_wc,1))*100

    print(f'  ══ Speech {i+1}: {title[:65]} ══')
    print(f'  Original: {orig_wc} words  →  Summary: {summ_wc} words  '
          f'(compression: {comp:.0f}%)')
    print(f'\n  EXTRACTIVE SUMMARY:')
    for j, sent in enumerate(summary, 1):
        print(f'  [{j}] {sent[:200]}')
    print()

print('─'*65)
print('ABSTRACTIVE SUMMARIZATION (conceptual):')
print('─'*65)
print('''
  Abstractive summarization generates NEW text that paraphrases
  the original. It uses sequence-to-sequence neural models:

  • Models: BART, T5, GPT (transformer-based)
  • Approach: Encoder reads input → Decoder generates summary
  • Unlike extractive: can rephrase, merge, and restructure

  Example (T5 API pattern):
    from transformers import pipeline
    summarizer = pipeline("summarization", model="t5-small")
    result = summarizer(text, max_length=130, min_length=30)
    print(result[0]['summary_text'])

  Comparison:
  ┌──────────────┬──────────────────────────┬────────────────────────┐
  │ Property     │ Extractive               │ Abstractive            │
  ├──────────────┼──────────────────────────┼────────────────────────┤
  │ Method       │ Select existing sentences│ Generate new sentences │
  │ Faithfulness │ Always factual           │ May hallucinate        │
  │ Fluency      │ Can feel disjointed      │ Reads naturally        │
  │ Implemented  │ ✅ TextRank above         │ Requires GPU/API       │
  └──────────────┴──────────────────────────┴────────────────────────┘
''')

TEXT SUMMARIZATION

EXTRACTIVE (TextRank):
Algorithm:
  1. Segment text into sentences
  2. TF-IDF vectorize each sentence
  3. Build cosine-similarity graph (sentences = nodes)
  4. PageRank to score each sentence by importance
  5. Return top-N sentences in original reading order

  ══ Speech 1: FACT SHEET:  United States – Argentina Relationship ══
  Original: 2831 words  →  Summary: 107 words  (compression: 96%)

  EXTRACTIVE SUMMARY:
  [1] Together, the leaders explored opportunities to strengthen the relationship between the United States and Argentina and partner to address global challenges, such as climate change, peacekeeping, refu
  [2] The leaders also discussed President Macri’s economic reform agenda, opportunities for expanding trade and investment, science and technology cooperation, and U.S. support for building Argentina’s cap
  [3] The countries will carry out further work through the United States-Argentina Binational Energy Working Group (BEWG) and the State Depart

---
## 🌐 Word Cloud
> **Syllabus:** Visualisation · Frequency Analysis · Content Analysis

In [34]:
# ══════════════════════════════════════════════════════
# CELL 22 ▸ WORD CLOUDS — Overall + Per Topic
# ══════════════════════════════════════════════════════
all_text = ' '.join(df['clean_text'].tolist())

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.patch.set_facecolor('#0f0f1a')
axes_flat = axes.flatten()

WC_COLORS = ['viridis','plasma','magma','inferno','cividis','cool']

# Overall word cloud
wc = WordCloud(width=700, height=400, background_color='#1a1a2e',
               colormap='plasma', max_words=80,
               collocations=False).generate(all_text)
axes_flat[0].imshow(wc, interpolation='bilinear')
axes_flat[0].axis('off')
axes_flat[0].set_title('All Development Speeches', fontweight='bold')

# Per-topic word clouds
for i, tid in enumerate(range(min(5, NUM_TOPICS)), 1):
    topic_text = ' '.join(
        df[df['lda_topic']==tid]['clean_text'].tolist()
    )
    if not topic_text.strip():
        axes_flat[i].axis('off'); continue
    wc = WordCloud(width=500, height=300,
                   background_color='#1a1a2e',
                   colormap=WC_COLORS[i%len(WC_COLORS)],
                   max_words=50,
                   collocations=False).generate(topic_text)
    axes_flat[i].imshow(wc, interpolation='bilinear')
    axes_flat[i].axis('off')
    axes_flat[i].set_title(f'Topic: {TOPIC_LABELS[tid][:30]}',
                            fontweight='bold', fontsize=9)

plt.suptitle('Word Clouds — Development Speeches by Topic',
             fontsize=14, color='white', fontweight='bold')
plt.tight_layout()
save_plot('17_wordclouds.png')

  💾  Saved → C:\Users\Thanooj Kumar G\obama_plots\17_wordclouds.png


---
## 🌍 Web Scraping
> **Syllabus:** HTML/CSS/XPath · BeautifulSoup · Scrapy · Selenium · Ethics · robots.txt · Captchas

In [35]:
# ══════════════════════════════════════════════════════
# CELL 23 ▸ WEB SCRAPING — Ethics, robots.txt, BeautifulSoup
# ══════════════════════════════════════════════════════
import urllib.robotparser

print('WEB SCRAPING PIPELINE\n')
print('STEP 1 — Ethics & robots.txt Compliance:')
print('─'*60)
print('''
  robots.txt is a file at <domain>/robots.txt that specifies
  which paths crawlers may or may not access.

  Ethical principles:
  ┌──────────────────────┬───────────────────────────────────────┐
  │ Principle            │ Implementation                        │
  ├──────────────────────┼───────────────────────────────────────┤
  │ robots.txt check     │ urllib.robotparser before any crawl   │
  │ Crawl rate           │ time.sleep(2-5s) between requests     │
  │ User-Agent header    │ Identify bot transparently            │
  │ No auth bypass       │ Only scrape public content            │
  │ Legal compliance     │ GDPR, CFAA, ToS of target site        │
  │ Data minimisation    │ Collect only what is needed           │
  └──────────────────────┴───────────────────────────────────────┘
''')

BASE_URL = 'https://www.americanrhetoric.com'
rp = urllib.robotparser.RobotFileParser()
rp.set_url(f'{BASE_URL}/robots.txt')
try:
    rp.read()
    paths = ['/Barack_Obama_Speeches.htm','/speeches/','/index.htm']
    print('  robots.txt check results:')
    for path in paths:
        ok = rp.can_fetch('*', BASE_URL+path)
        print(f'  [{"ALLOWED" if ok else "BLOCKED":>7}] {path}')
except Exception as e:
    print(f'  [NOTE] Could not reach robots.txt: {e}')
    print('  (This is expected if running offline)')

print()
print('STEP 2 — BeautifulSoup HTML Parsing Demo:')
print('─'*60)
# Demonstrate on URLs from our CSV if available
sample_url = str(df['url'].iloc[0]) if 'url' in df.columns else ''
print(f'\n  Sample URL from dataset: {sample_url[:70]}')

HTML_DEMO = '''
<html>
  <head><title>Obama Speech</title></head>
  <body>
    <div class="transcript">
      <h1>State of the Union 2012</h1>
      <p class="speech-text">America is back. Anyone who tells you
         otherwise is selling something.</p>
      <p class="speech-text">We need to invest in education,
         clean energy, and infrastructure.</p>
    </div>
  </body>
</html>
'''

soup = BeautifulSoup(HTML_DEMO, 'html.parser')
print(f'\n  Parsed HTML elements:')
print(f'  Title tag     : {soup.title.string}')
print(f'  H1 tag        : {soup.h1.string}')
print(f'  All <p> tags  :')
for i, p in enumerate(soup.find_all('p'), 1):
    print(f'    [{i}] {p.text[:80]}')
print(f'\n  CSS selector .speech-text:')
for p in soup.select('.speech-text'):
    print(f'    → {p.text[:80]}')

print()
print('STEP 3 — Polite Crawl Rate:')
print('─'*60)
print('''
  def polite_delay(min_sec=2.0, max_sec=5.0):
      delay = random.uniform(min_sec, max_sec)
      time.sleep(delay)
      # Randomised delay avoids pattern detection
      # and reduces server load

  HEADERS = {
      "User-Agent": "AcademicBot/1.0 (contact: you@uni.edu)"
  }
  # Always identify your bot transparently
''')

print('STEP 4 — Handling Dynamic Content (Selenium pattern):')
print('─'*60)
print('''
  BeautifulSoup → Static HTML (works on our dataset)
  Scrapy        → Large-scale crawls with built-in pipelines
  Selenium      → JavaScript-rendered pages, SPAs

  from selenium import webdriver
  driver = webdriver.Chrome()
  driver.get(url)
  driver.implicitly_wait(3)  # wait for JS to render
  html = driver.page_source
  soup = BeautifulSoup(html, "html.parser")

  Captchas     → Use time.sleep, rotate user-agents, use proxies
  Proxies      → Rotate IP with requests-ip-rotator or Scrapy middlewares
  Rate limiting→ Exponential backoff on 429/503 responses
''')

# ── Plot: URL domain frequency ──
if 'url' in df.columns:
    from urllib.parse import urlparse
    domains = df['url'].dropna().apply(
        lambda u: urlparse(str(u)).netloc
    ).value_counts().head(8)

    if len(domains) > 0:
        fig, ax = plt.subplots(figsize=(11, 4))
        fig.patch.set_facecolor('#0f0f1a')
        ax.barh(range(len(domains)), domains.values,
                color=PALETTE[:len(domains)])
        ax.set_yticks(range(len(domains)))
        ax.set_yticklabels(domains.index, fontsize=9)
        ax.invert_yaxis()
        ax.set_title('Source Domains in Dataset (Web Scraping Origin)',
                     fontweight='bold')
        ax.set_xlabel('Count')
        plt.tight_layout()
        save_plot('18_source_domains.png')

WEB SCRAPING PIPELINE

STEP 1 — Ethics & robots.txt Compliance:
────────────────────────────────────────────────────────────

  robots.txt is a file at <domain>/robots.txt that specifies
  which paths crawlers may or may not access.

  Ethical principles:
  ┌──────────────────────┬───────────────────────────────────────┐
  │ Principle            │ Implementation                        │
  ├──────────────────────┼───────────────────────────────────────┤
  │ robots.txt check     │ urllib.robotparser before any crawl   │
  │ Crawl rate           │ time.sleep(2-5s) between requests     │
  │ User-Agent header    │ Identify bot transparently            │
  │ No auth bypass       │ Only scrape public content            │
  │ Legal compliance     │ GDPR, CFAA, ToS of target site        │
  │ Data minimisation    │ Collect only what is needed           │
  └──────────────────────┴───────────────────────────────────────┘

  robots.txt check results:
  [BLOCKED] /Barack_Obama_Speeches.htm
  [BLO

---
## 💾 Save All Plots & Download
> All plots are saved to your `obama_plots/` folder automatically after each cell.
> Run this cell to create a ZIP and download everything at once.

In [36]:
# ══════════════════════════════════════════════════════
# CELL 27 ▸ SAVE ALL PLOTS + ZIP DOWNLOAD
# ══════════════════════════════════════════════════════
import zipfile

ZIP_NAME = 'obama_nlp_plots.zip'
saved_files = sorted([
    f for f in os.listdir(PLOT_DIR)
    if f.endswith('.png')
])

print('═'*60)
print('  ANALYSIS COMPLETE — All Plots Generated')
print('═'*60)
print(f'\n  Output folder : {os.path.abspath(PLOT_DIR)}/')
print(f'  Total plots   : {len(saved_files)}\n')

PLOT_DESCRIPTIONS = {
    '01_dataset_overview.png'           : 'Document types + speeches by year',
    '02_token_frequency.png'            : 'Top 30 raw token frequencies',
    '03_stopword_removal.png'           : 'Before vs after stop word removal',
    '04_stem_vs_lemma.png'              : 'Stemming vs lemmatization comparison',
    '05_bow_top_terms.png'              : 'Bag of Words top 20 terms',
    '06_tfidf_heatmap.png'              : 'TF-IDF term × speech heatmap',
    '07_ngrams.png'                     : 'Unigram vs bigram vs trigram analysis',
    '08_word2vec_tsne.png'              : 'Word2Vec embeddings t-SNE projection',
    '09_classification_accuracy.png'    : 'NB vs SVM vs Logistic Regression',
    '10_sentiment_analysis.png'         : 'VADER sentiment distribution + arc',
    '11_aspect_sentiment.png'           : 'Aspect-based sentiment by policy domain',
    '12_lda_topics.png'                 : 'LDA topic distribution + word weights',
    '13_lsa_document_space.png'         : 'LSA document space projection',
    '14_kmeans_clustering.png'          : 'K-Means elbow + cluster sizes',
    '15_hierarchical_dendrogram.png'    : 'Hierarchical clustering dendrogram',
    '16_ner_analysis.png'               : 'Named entity recognition frequency',
    '17_wordclouds.png'                 : 'Word clouds overall + per topic',
    '18_source_domains.png'             : 'Source domains (web scraping origin)',
    '19_web_analytics_kpis.png'         : 'Simulated web traffic KPIs dashboard',
    '20_similarity_recommendation.png'  : 'Speech similarity recommendation matrix',
    '21_risk_competitive_intelligence.png': 'Risk signals + competitive bigrams'
}

for f in saved_files:
    desc = PLOT_DESCRIPTIONS.get(f, '')
    size = os.path.getsize(os.path.join(PLOT_DIR, f)) / 1024
    print(f'  ✅  {f:<42} {size:>6.0f} KB   {desc}')

# Create ZIP
with zipfile.ZipFile(ZIP_NAME, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in saved_files:
        zf.write(os.path.join(PLOT_DIR, f), f)
zip_size = os.path.getsize(ZIP_NAME)/1024
print(f'\n  📦  ZIP created: {ZIP_NAME} ({zip_size:.0f} KB)')

# Auto-download in Colab
try:
    from google.colab import files
    files.download(ZIP_NAME)
    print('  ⬇️   Download started automatically (Colab).')
except ImportError:
    print(f'  📂  Local Jupyter: ZIP saved to {os.path.abspath(ZIP_NAME)}')
    print(f'      Individual plots in: {os.path.abspath(PLOT_DIR)}/')

print(f'''
══════════════════════════════════════════════════════════════
  SYLLABUS COVERAGE SUMMARY
══════════════════════════════════════════════════════════════
  ✅  Structured vs Unstructured Data
  ✅  Tokenization (Word + Sentence)
  ✅  Stop Word Removal (with full list of removed words)
  ✅  Stemming (Porter Stemmer)
  ✅  Lemmatization (WordNet) — compared to Stemming
  ✅  Bag of Words (BoW)
  ✅  TF-IDF Vectorization + Heatmap
  ✅  N-Grams (Unigrams, Bigrams, Trigrams)
  ✅  Word2Vec Embeddings + t-SNE
  ✅  GloVe (conceptual comparison)
  ✅  Naive Bayes Text Classification
  ✅  SVM (LinearSVC) Classification
  ✅  Logistic Regression Classification
  ✅  VADER Rule-Based Sentiment Analysis
  ✅  Aspect-Based Sentiment Analysis
  ✅  LDA Topic Modelling (Gensim)
  ✅  LSA (TruncatedSVD)
  ✅  K-Means Clustering + Elbow Method
  ✅  Hierarchical Clustering + Dendrogram
  ✅  Rule-Based NER (NLTK)
  ✅  ML-Based NER (spaCy)
  ✅  Extractive Summarization (TextRank)
  ✅  Abstractive Summarization (conceptual)
  ✅  Word Clouds (Overall + Per Topic)
  ✅  Web Scraping (BeautifulSoup, robots.txt, Selenium pattern)
  ✅  Web Analytics KPIs (Page Views, Bounce Rate, CVR)
  ✅  Recommendation System (Content-Based)
  ✅  Customer Analytics & Personalization
  ✅  Competitive Intelligence
  ✅  Risk Management & Signal Detection
══════════════════════════════════════════════════════════════
''')

════════════════════════════════════════════════════════════
  ANALYSIS COMPLETE — All Plots Generated
════════════════════════════════════════════════════════════

  Output folder : C:\Users\Thanooj Kumar G\obama_plots/
  Total plots   : 18

  ✅  01_dataset_overview.png                        83 KB   Document types + speeches by year
  ✅  02_token_frequency.png                         99 KB   Top 30 raw token frequencies
  ✅  03_stopword_removal.png                        83 KB   Before vs after stop word removal
  ✅  04_stem_vs_lemma.png                           75 KB   Stemming vs lemmatization comparison
  ✅  05_bow_top_terms.png                           88 KB   Bag of Words top 20 terms
  ✅  06_tfidf_heatmap.png                          169 KB   TF-IDF term × speech heatmap
  ✅  07_ngrams.png                                 122 KB   Unigram vs bigram vs trigram analysis
  ✅  08_word2vec_tsne.png                           72 KB   Word2Vec embeddings t-SNE projection
  ✅  09_class